# Direct Silver Crypto OHLC 1d

Thin notebook entrypoint for the Silver market crypto OHLC path.

This notebook delegates Bronze table reads, key-level deduplication, `product_id` parsing, Silver DQ checks, quarantine writes, and Silver Delta merge logic to `src/lakehouse`.

Execution modes:

- `backfill`: explicit inclusive date range
- `incremental`: rolling lookback window ending on the latest completed UTC day

The Bronze, Silver, and Silver quarantine tables must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

Default product set: `BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD`.

In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.silver import run_silver_crypto_ohlc_1d

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("product_ids", "BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "3")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
result = run_silver_crypto_ohlc_1d(
    spark=spark,
    raw_product_ids=dbutils.widgets.get("product_ids").strip(),
    mode=dbutils.widgets.get("mode").strip().lower(),
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
    run_id=dbutils.widgets.get("run_id").strip(),
    catalog=dbutils.widgets.get("catalog").strip() or "market_macro",
    display_fn=display,
)

result.as_dict()
